<a href="https://colab.research.google.com/github/zencolab/colab/blob/main/IndexTTS2_5_L4_Colab_20261007.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IndexTTS-2.5 · Google Colab Pro L4

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zencolab/colab/blob/main/IndexTTS2_5_L4_Colab.ipynb)

模型：**IndexTeam/IndexTTS-2.5**（目前官方最新模型）。代码默认固定到官方 `main` 最新提交 `d9e41aa`（包含 v2.5.0 之后的修复：结尾咔哒声淡出、`do_sample` 透传、推理提速、WebUI 预设/`--share` 等），也可改回 `v2.5.0`。

**本版修复的 Colab 报错**：Colab 预设了 `UV_PRERELEASE=`（空值）、`UV_CONSTRAINT`、`UV_SYSTEM_PYTHON`、`MPLBACKEND=module://matplotlib_inline...`、`PYTHONPATH` 等环境变量，会导致 `uv sync` 直接报 `a value is required for '--prerelease'`，或在 uv 虚拟环境里导入 matplotlib 失败。现在所有 `uv` 命令都在清理后的环境中运行。

> 先在「运行时 → 更改运行时类型」选择 GPU/L4。只使用本人或已获明确授权的声音，并阅读上游许可与免责声明。


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

#@markdown ### 基本设置
CODE_REF = "d9e41aac89fd00b3d71497fddb287b7f24613712" #@param {type:"string"}
#@markdown ↑ 官方仓库的提交 SHA 或标签（如 `v2.5.0`）
USE_DRIVE = False #@param {type:"boolean"}
HF_ENDPOINT = "" #@param {type:"string"}
REPO = Path("/content/index-tts")

# Colab 注入的这些变量会破坏 uv / 子进程虚拟环境，统一清理
_BAD_PREFIX = ("UV_",)
_BAD_KEYS = {"MPLBACKEND", "PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV", "CONDA_PREFIX", "PIP_CONSTRAINT"}
def clean_env():
    env = {k: v for k, v in os.environ.items() if k not in _BAD_KEYS and not k.startswith(_BAD_PREFIX)}
    env.update(UV_LINK_MODE="copy", MPLBACKEND="Agg", PYTHONUNBUFFERED="1",
               HF_HUB_DISABLE_PROGRESS_BARS="0", TOKENIZERS_PARALLELISM="false")
    if HF_ENDPOINT.strip():
        env["HF_ENDPOINT"] = HF_ENDPOINT.strip().rstrip("/")
    return env

def run(cmd, cwd=None):
    """实时显示子进程输出（Colab 不会显示直接写到 fd 的子进程输出）。"""
    cmd = list(map(str, cmd))
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=str(cwd) if cwd else None, env=clean_env(),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    while True:
        chunk = os.read(p.stdout.fileno(), 4096)
        if not chunk:
            break
        sys.stdout.write(chunk.decode("utf-8", "replace")); sys.stdout.flush()
    if p.wait():
        raise subprocess.CalledProcessError(p.returncode, cmd)

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     text=True, capture_output=True)
if smi.returncode:
    raise RuntimeError("未检测到 NVIDIA GPU，请切换到 Colab GPU 运行时。")
print(smi.stdout.strip())
if "L4" not in smi.stdout:
    print("⚠️ 当前不是 L4，仍可尝试运行（建议显存 ≥ 16GB）。")


## 1. 安装

首次安装约 3–6 分钟。uv 会自动下载项目要求的 Python 3.11 并按官方 `uv.lock` 安装（CUDA 12.8 版 PyTorch 2.8）。不安装易在 Colab 编译失败的 DeepSpeed/FlashAttention。


In [ ]:
run(["apt-get", "-qq", "update"])
run(["apt-get", "-qq", "install", "-y", "git", "ffmpeg"])

if not (REPO / ".git").exists():
    if REPO.exists(): shutil.rmtree(REPO)
    run(["git", "init", "-q", REPO])
    run(["git", "remote", "add", "origin", "https://github.com/index-tts/index-tts.git"], REPO)
run(["git", "fetch", "-q", "--depth", "1", "origin", CODE_REF], REPO)
run(["git", "checkout", "-q", "--force", "FETCH_HEAD"], REPO)
run(["git", "log", "-1", "--format=%h %cs %s"], REPO)

run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"])
UV = shutil.which("uv") or "/usr/local/bin/uv"
run([UV, "--version"])
run([UV, "sync", "--extra", "webui", "--frozen"], REPO)
run([UV, "run", "--no-sync", "python", "-c",
     "import torch;print('torch',torch.__version__,'cuda',torch.cuda.is_available())"], REPO)


## 2. 下载 IndexTTS-2.5 模型

开启 `USE_DRIVE` 后模型保存到 `MyDrive/AI-Models/IndexTTS-2.5`，下次会话可直接复用。


In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    MODEL_DIR = Path("/content/drive/MyDrive/AI-Models/IndexTTS-2.5")
else:
    MODEL_DIR = REPO / "checkpoints"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

program = f"""
from indextts.utils.model_download import snapshot_download, ensure_config_available, ensure_models_available
m = {str(MODEL_DIR)!r}
snapshot_download("IndexTeam/IndexTTS-2.5", local_dir=m)
ensure_config_available(m, version="2.5")
ensure_models_available(m)
"""
run([UV, "run", "--no-sync", "python", "-c", program], REPO)

required = ["config.yaml", "gpt.pth", "s2mel.pth", "codec.pth",
            "multilingual_zh_ja_yue_char_del.tiktoken", "wav2vec2bert_stats.pt"]
missing = [x for x in required if not (MODEL_DIR / x).is_file()]
if missing:
    raise FileNotFoundError("模型不完整，请重跑本单元格：" + ", ".join(missing))
run([UV, "run", "--no-sync", "python", "tools/gpu_check.py"], REPO)
print("✅ 模型就绪：", MODEL_DIR)


## 3. 启动官方 WebUI（推荐）

运行完第 1、2 节后直接运行本单元格即可，在网页里上传参考音频并生成，无需运行第 4 节。会实时显示加载日志；启动完成后输出 Colab 内置访问链接（同时尝试生成 `https://xxxx.gradio.live` 公网链接）。单元格会持续运行，用完点击停止。L4（24GB）会额外加载 QwenEmotion 文本情感模型，支持“用文本描述情感”。


In [ ]:
LAUNCH_WEBUI = True #@param {type:"boolean"}
PORT = 7860
if LAUNCH_WEBUI:
    import socket, threading, time
    from google.colab.output import eval_js
    log_path = Path("/content/webui.log")
    cmd = [UV, "run", "--no-sync", "python", "webui.py", "--version", "2.5", "--model_dir", str(MODEL_DIR),
           "--fp16", "--share", "--host", "0.0.0.0", "--port", str(PORT)]
    print("$", " ".join(cmd), "\n首次启动需加载模型和示例音频，约 2–5 分钟，日志如下：\n", flush=True)
    proc = subprocess.Popen(cmd, cwd=str(REPO), env=clean_env(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    def pump():
        with open(log_path, "wb") as f:
            while True:
                chunk = os.read(proc.stdout.fileno(), 4096)
                if not chunk: break
                f.write(chunk); f.flush()
                sys.stdout.write(chunk.decode("utf-8", "replace")); sys.stdout.flush()
    threading.Thread(target=pump, daemon=True).start()
    shown = False
    try:
        while proc.poll() is None:
            if not shown:
                with socket.socket() as sk:
                    if sk.connect_ex(("127.0.0.1", PORT)) == 0:
                        url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
                        print(f"\n✅ WebUI 已启动。Colab 内置链接（无需 gradio.live）：{url}\n"
                              "   gradio.live 公网链接若生成成功，会出现在上方日志中。\n", flush=True)
                        shown = True
            time.sleep(2)
        print(f"\n❌ WebUI 已退出（code {proc.returncode}），请查看上方日志或 {log_path}")
    except KeyboardInterrupt:
        proc.terminate(); print("已停止 WebUI")
else:
    print("已跳过 WebUI")
